# analysis_market_regime_builder_v1

目标：验证 `src/analysis` 中的 market regime builder 是否能从真实 SOXL/SOXX/VIX 面板生成稳定 `regime_frame`，并通过 `build_market_regime_analysis_report` 接入 qlib/backtest 的 regime/window 模型差异分析链路。

本 notebook 只保存本次运行参数、sidecar 和中文结论；可复用逻辑保留在 `src/analysis`。

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

REPO_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'AGENTS.md').exists()
)
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
from IPython.display import Markdown, display

from src.analysis import (
    MarketRegimeConfig,
    analyze_model_effects_by_regime_and_window,
    attach_market_regimes,
    build_analysis_windows,
    build_market_regime_analysis_report,
    build_market_regime_frame,
    build_regime_window_analysis_report,
    compare_models,
    compute_regime_metrics,
    compute_window_metrics,
    write_analysis_report,
)
from src.core import DatasetSnapshot
from src.data import load_soxl_support_frame

CARD = Path('research/analysis_market_regime_builder_v1')
CARD.mkdir(parents=True, exist_ok=True)
RUN_START = '2024-01-01'
RUN_END = '2026-06-10'


## 1. 生成 market regime frame

In [2]:
market = load_soxl_support_frame(start=RUN_START, end=RUN_END)
config = MarketRegimeConfig(
    trend_window=21,
    volatility_window=21,
    drawdown_window=63,
    volume_window=20,
    high_vix_threshold=30.0,
    low_vix_threshold=20.0,
    stress_drawdown_threshold=-0.10,
    bear_drawdown_threshold=-0.03,
    uptrend_threshold=0.03,
    downtrend_threshold=-0.03,
    sideways_trend_abs_threshold=0.01,
    high_volatility_threshold=0.045,
    low_volatility_threshold=0.020,
    volume_shock_z_threshold=2.0,
)
regimes = build_market_regime_frame(market, config=config)
regime_counts = regimes['regime'].value_counts().rename_axis('regime').reset_index(name='row_count')
regimes.to_csv(CARD / 'market_regimes.csv', index=False)
regime_counts.to_csv(CARD / 'market_regime_counts.csv', index=False)
regime_counts


,regime,row_count
0,stress,374
1,high_vol_uptrend,161
2,bull,49
3,bear,17
4,volume_shock,5
5,neutral,4
6,sideways,1
7,high_vol_downtrend,1


## 2. 接入日度模型回测样例

In [3]:
work = market.merge(regimes[['date', 'regime']], on='date', how='left')
work['soxl_return'] = work['close'].astype(float).pct_change().fillna(0.0)
work['benchmark_return'] = work['soxx_close'].astype(float).pct_change().fillna(0.0)

exposure_profiles = {
    'ridge_mse_sample': {'default': 0.35, 'bull': 0.40, 'high_vol_uptrend': 0.38, 'stress': 0.22, 'bear': 0.24, 'sideways': 0.25, 'volume_shock': 0.30},
    'gcn_mse_sample': {'default': 0.45, 'bull': 0.62, 'high_vol_uptrend': 0.70, 'stress': 0.40, 'bear': 0.36, 'sideways': 0.30, 'volume_shock': 0.52},
    'rgcn_mse_sample': {'default': 0.38, 'bull': 0.50, 'high_vol_uptrend': 0.48, 'stress': 0.12, 'bear': 0.18, 'sideways': 0.25, 'volume_shock': 0.28},
}

frames = []
benchmark_nav = (1.0 + work['benchmark_return']).cumprod()
for model, profile in exposure_profiles.items():
    exposure = work['regime'].map(lambda value: profile.get(str(value), profile['default'])).astype(float)
    strategy_return = work['soxl_return'] * exposure
    nav = (1.0 + strategy_return).cumprod()
    frames.append(
        pd.DataFrame(
            {
                'date': work['date'],
                'model': model,
                'nav': nav,
                'benchmark_nav': benchmark_nav,
                'position': exposure,
                'turnover': exposure.diff().abs().fillna(0.0),
            }
        )
    )

daily_backtest = pd.concat(frames, ignore_index=True)
daily_backtest = attach_market_regimes(daily_backtest, regimes, overwrite=True)
daily_backtest.to_csv(CARD / 'sample_daily_backtest.csv', index=False)
daily_backtest.head()


,date,model,nav,benchmark_nav,position,turnover,market_regime,regime_tags,regime_reason,vix,trend,drawdown,realized_vol,volume_z,relative_strength,regime
0,2024-01-02,ridge_mse_sample,1.000000,1.000000,0.25,0.00,sideways,low_vix|low_volatility|sideways,低波动震荡 regime：检查模型是否过度交易或信号噪声过高。,13.20,0.000000,0.000000,0.000000,0.000000,0.0,sideways
1,2024-01-03,ridge_mse_sample,0.977657,0.978518,0.35,0.10,high_vol_downtrend,low_vix|drawdown|high_volatility|downtrend,高波动下行 regime：波动和下行趋势同时出现，检查回撤控制。,14.04,-0.032971,-0.063837,0.045140,-0.707107,0.0,high_vol_downtrend
2,2024-01-04,ridge_mse_sample,0.971668,0.970289,0.24,0.11,bear,low_vix|drawdown|downtrend,下行或回撤 regime：优先检查亏损控制和暴露收缩。,14.13,-0.039189,-0.087732,0.032132,0.297571,0.0,bear
3,2024-01-05,ridge_mse_sample,0.974859,0.975673,0.24,0.00,bear,low_vix|drawdown,下行或回撤 regime：优先检查亏损控制和暴露收缩。,13.35,-0.019660,-0.075250,0.034069,0.158341,0.0,bear
4,2024-01-08,ridge_mse_sample,1.010575,1.007455,0.38,0.14,high_vol_uptrend,low_vix|high_volatility|uptrend,高波动上行 regime：趋势强但波动高，检查追涨和仓位弹性。,13.08,0.059003,0.000000,0.059421,0.054318,0.0,high_vol_uptrend


## 3. 运行现有 analysis report 链路

In [4]:
windows = build_analysis_windows(
    start=str(work['date'].min().date()),
    end=str(work['date'].max().date()),
    custom_windows={
        'recent_2025_2026': ('2025-01-01', RUN_END),
        'drawdown_review_2024': ('2024-04-01', '2024-12-31'),
    },
)
dataset = DatasetSnapshot(
    name='soxl_support_market_regime_builder_v1',
    universe=('SOXL', 'SOXX', 'VIX'),
    start=str(work['date'].min().date()),
    end=str(work['date'].max().date()),
    calendar_path='data/qlib/soxl/calendars/day.txt',
    data_paths=('data/processed/panels/soxl_soxx_vix_full.csv',),
    metadata={
        'row_count': int(len(work)),
        'regime_schema_version': 'analysis.market_regime.v1',
        'purpose': '验证 market regime builder 接入 qlib/backtest analysis report。',
    },
)
report = build_market_regime_analysis_report(
    artifact_id='analysis_market_regime_builder_v1',
    daily_backtest=daily_backtest,
    market_frame=market,
    windows=windows,
    baseline_model='ridge_mse_sample',
    config=config,
    summary={
        'status': 'market_regime_builder_completed',
        'analysis_direction': 'analysis',
        'note': '中文 review：market regime 生成、重分类和模型差异 report 留在 src/analysis，skill 只提供候选切法。',
        'regime_counts': regime_counts.to_dict(orient='records'),
    },
    metadata={'source': 'research/analysis_market_regime_builder_v1'},
)
model_comparison = report.tables['model_comparison']
effect_summary = report.summary.get('analysis_summary', {})
export = write_analysis_report(report, CARD / 'analysis_report', dataset=dataset)
summary = {
    'schema_version': 'research.analysis_market_regime_builder_v1',
    'status': 'completed',
    'start': dataset.start,
    'end': dataset.end,
    'market_regime_rows': int(len(regimes)),
    'regime_count': int(regimes['regime'].nunique()),
    'regime_counts': regime_counts.to_dict(orient='records'),
    'model_count': int(daily_backtest['model'].nunique()),
    'window_count': int(len(windows)),
    'analysis_report': 'research/analysis_market_regime_builder_v1/analysis_report',
    'top_weakness': effect_summary.get('weakest_segments', [])[:3],
    'export': export,
}
(CARD / 'analysis_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'schema_version': 'research.analysis_market_regime_builder_v1',
 'status': 'completed',
 'start': '2024-01-02',
 'end': '2026-06-10',
 'market_regime_rows': 612,
 'regime_count': 8,
 'regime_counts': [{'regime': 'stress', 'row_count': 374},
  {'regime': 'high_vol_uptrend', 'row_count': 161},
  {'regime': 'bull', 'row_count': 49},
  {'regime': 'bear', 'row_count': 17},
  {'regime': 'volume_shock', 'row_count': 5},
  {'regime': 'neutral', 'row_count': 4},
  {'regime': 'sideways', 'row_count': 1},
  {'regime': 'high_vol_downtrend', 'row_count': 1}],
 'model_count': 3,
 'window_count': 8,
 'analysis_report': 'research/analysis_market_regime_builder_v1/analysis_report',
 'top_weakness': [{'diagnosis': '弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。',
   'model': 'gcn_mse_sample',
   'relative_return_pct_delta': -3.284316422888689,
   'segment': 'drawdown_review_2024',
   'segment_type': 'window',
   'stability_score_delta': -22.232299524889875},
  {'diagnosis': '弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 r

In [5]:
top_model = model_comparison.iloc[0]
regime_text = ', '.join(f"{row.regime}={int(row.row_count)}" for row in regime_counts.itertuples(index=False))
weakest = effect_summary.get('weakest_segments', [])[:1]
weakness_text = weakest[0]['segment'] if weakest else '无明显弱势分段'

display(Markdown(f'''## 中文结论

- 做了什么：从真实 SOXL/SOXX/VIX 日频面板生成 `analysis.market_regime.v1` 的 `market_regimes.csv`，并通过 `build_market_regime_analysis_report` 重分类样例日度模型回测，复用 qlib/regime/window analysis report 链路。
- 核心指标：覆盖 `{summary['start']}` 到 `{summary['end']}`，regime 行数 `{summary['market_regime_rows']}`，regime 类型 `{summary['regime_count']}`，样例模型数 `{summary['model_count']}`，窗口数 `{summary['window_count']}`。
- regime 差异：当前样例分布为 `{regime_text}`；最弱分段优先检查 `{weakness_text}`。
- 是否晋升：晋升为框架能力。`build_market_regime_analysis_report` 是 notebook/dashboard/schedule 的统一入口；`build_market_regime_frame` 和 `attach_market_regimes` 保留为底层组件；skill 只用于发散候选 regime 设计，不保存业务状态。
- 风险：本 notebook 的模型回测是接口验证样例，不代表真实模型业绩；真实模型比较应使用已训练模型的 `daily_backtest` 或 qlib recorder artifact。
- 下一步：把真实 QuantBench ledger 的模型回测接入该 `regime_frame`，并在 dashboard 中展示 model-regime-window 稳定性差异。

Sidecar：`market_regimes.csv`、`market_regime_counts.csv`、`sample_daily_backtest.csv`、`analysis_report/market_regimes.csv`、`analysis_report/analysis_summary.json`、`analysis_report/analysis_manifest.json`。
'''))


## 中文结论

- 做了什么：从真实 SOXL/SOXX/VIX 日频面板生成 `analysis.market_regime.v1` 的 `market_regimes.csv`，并通过 `build_market_regime_analysis_report` 重分类样例日度模型回测，复用 qlib/regime/window analysis report 链路。
- 核心指标：覆盖 `2024-01-02` 到 `2026-06-10`，regime 行数 `612`，regime 类型 `8`，样例模型数 `3`，窗口数 `8`。
- regime 差异：当前样例分布为 `stress=374, high_vol_uptrend=161, bull=49, bear=17, volume_shock=5, neutral=4, sideways=1, high_vol_downtrend=1`；最弱分段优先检查 `drawdown_review_2024`。
- 是否晋升：晋升为框架能力。`build_market_regime_analysis_report` 是 notebook/dashboard/schedule 的统一入口；`build_market_regime_frame` 和 `attach_market_regimes` 保留为底层组件；skill 只用于发散候选 regime 设计，不保存业务状态。
- 风险：本 notebook 的模型回测是接口验证样例，不代表真实模型业绩；真实模型比较应使用已训练模型的 `daily_backtest` 或 qlib recorder artifact。
- 下一步：把真实 QuantBench ledger 的模型回测接入该 `regime_frame`，并在 dashboard 中展示 model-regime-window 稳定性差异。

Sidecar：`market_regimes.csv`、`market_regime_counts.csv`、`sample_daily_backtest.csv`、`analysis_report/market_regimes.csv`、`analysis_report/analysis_summary.json`、`analysis_report/analysis_manifest.json`。


## 中文结论

- 做了什么：从真实 SOXL/SOXX/VIX 日频面板生成 `analysis.market_regime.v1` 的 `market_regimes.csv`，并通过 `build_market_regime_analysis_report` 重分类样例日度模型回测，复用 qlib/regime/window analysis report 链路。
- 核心指标：覆盖 `2024-01-02` 到 `2026-06-10`，regime 行数 `612`，regime 类型 `8`，样例模型数 `3`，窗口数 `8`。
- regime 差异：当前样例分布为 `stress=374, high_vol_uptrend=161, bull=49, bear=17, volume_shock=5, neutral=4, sideways=1, high_vol_downtrend=1`；最弱分段优先检查 `drawdown_review_2024`。
- 是否晋升：晋升为框架能力。`build_market_regime_analysis_report` 是 notebook、dashboard、schedule 的统一入口；`build_market_regime_frame` 和 `attach_market_regimes` 保留为底层组件；skill 只用于发散候选 regime 设计，不保存业务状态。
- 风险：本 notebook 的模型回测是接口验证样例，不代表真实模型业绩；真实模型比较应使用已训练模型的 `daily_backtest` 或 qlib recorder artifact。
- 下一步：把真实 QuantBench ledger 的模型回测接入该 `regime_frame`，并在 dashboard 中展示 model-regime-window 稳定性差异。

Sidecar：`market_regimes.csv`、`market_regime_counts.csv`、`sample_daily_backtest.csv`、`analysis_report/market_regimes.csv`、`analysis_report/analysis_summary.json`、`analysis_report/analysis_manifest.json`。
